FRAME TO FRAME INFERENCE & EVALUATION

In [1]:
# imports from run_test
from __future__ import print_function
from __future__ import division
import time, os, sys
import numpy as np
import soundfile as sf
from spicy import signal
import pickle
import warnings
import gzip
import scipy.io
from scipy.io import wavfile
from scipy.io import loadmat
import matplotlib.pyplot as plt

warnings.simplefilter('ignore')

import torch
import torch.utils.data as data

# 8k Net trained
#workspace_dir = '/home/adelval/BTS/TFM/afterburner8k/'
# 16k Net trained
workspace_dir = '/home/adelval/BTS/TFM/test/'

sys.path.append(workspace_dir + 'src/net')
sys.path.append(workspace_dir + 'src/train')
sys.path.append(workspace_dir + 'src/eval')
from vvtk_net.v1.utils import *
from vvtk_net.v1.transforms import *
from vvtk_net.v1.transforms_fe import *
from vvtk_net.v1.datafeed import *
from vvtk_net.v1.layers_pytorch import *
from vvtk_net.config import Configuration
from eval_utils import *

In [2]:

def read_audio(f):
    x, fs = sf.read(f)
    return np.array(x * 2**15, dtype=np.int16), fs

In [3]:

def offset(x):
    return signal.lfilter([1.0, -1.0],[1, -0.99899,], x)

def preemphasis(x):
    #return signal.lfilter([1.0, -0.97],1, x)    
    x0 = x[0] * (1.0 - 0.97)
    for i in reversed(range(len(x))): 
        x[i] = x[i] - x[i-1] * 0.97
    x[0] = x0
    return x

# Divide x into overlapping frames of fixed length without extending to slide last frame
def windowing2(x, fs=16000, Ns=0.025, Ms=0.010, Mw=20):
    # Mw limit the number of windows
    N = int(Ns * fs)            # Number of samples in each window 640
    M = int(Ms * fs)            # Step size (number of samples between window starts) 160
    n = (len(x) + M - 1) // M   # Number of frames 23
    print("Number of frames", n)    
    # T = (n - 1) * M + N         # Total signal length needed to fit the frames
    xa = x.copy()
    # Se ignora el padding porque la ventana se deslizará
    # if T > len(x):
    #     print("rellena con ceros")
    #     xa.resize(T, refcheck=False)    # Pad the signal with zeros
    m = np.arange(0, Mw*M, M)          # Starting indices of each frame
    ind = np.arange(N).reshape(-1, 1) + m.reshape(1, -1) # 2D matrix whit rows correspnding to the samples whitin a window and columns the starting indeces each time
    xa = xa[ind.astype(int).T].astype(np.float32)
    # print(f'Window frames {xa[19,:5]}')

    return xa

def hamming(X):
    w = np.hamming(X.shape[1])
    return X * w

def fft(X, NFFT=None):
    if NFFT is None:
        NFFT = int(2 ** np.ceil(np.log2(X.shape[1])))
    Xfreq = np.abs(np.fft.fft(X, NFFT, axis=1)) # Computes the FFT along each row of X
    Xfreq = np.asarray(Xfreq, dtype=np.float32)
    return Xfreq[:, :(NFFT // 2)] # Return Half the Spectrum, only the first half is meaningful for real-valued signals


def frame_fft(data, fs, w, nfft, max_windows):
    
    N =[int(wi * fs) for wi in w]
    F =[int(2 ** np.ceil(np.log2( Ni)))//2 for Ni in N]

    x = offset(data)

    # Emphasis to increase the amplitude of high freq
    x = preemphasis(x)
    print(f'0 frame   {x[:10]}')
    print(f'1 frame   {x[1*int(m*fs):1*int(m*fs)+10]}')
    print(f'2 frame   {x[2*int(m*fs):2*int(m*fs)+10]}')
    print(f'3 frame   {x[3*int(m*fs):3*int(m*fs)+10]}')
    print(f'20 frame  {x[19*int(m*fs):19*int(m*fs)+10]}')

    X = hamming(windowing2(x, fs=fs, Ns=w[0], Ms=m, Mw=max_windows))
    # Dimesions of X are (N,n) having in row the signal windowed with hamming of win length
    Xfft = fft(X, nfft[0])
    print(f"la shape de Xfft es {Xfft.shape}")
    X = Xfft * Xfft # Power spectrum
    X = np.asarray(X, dtype=np.float32)
    
    return X    # Returns the PSD of the frame processed


# Log scale for PSD
def log_scale(frame_psd):
    scale=1.
    eps=1e-8
    x = frame_psd    
    x = np.abs(x)
    x = scale * np.log10(x + eps)  
    # print("Vector con Power Spectral Density in log scale del frame: \n", x[:10])

    return x   


In [4]:
#FB MFCC Filter Bank Mel-Frequency Cepstral Coefficients
def fb_etsi(F, B, fs):
    StFreq = 64.0
    fb = np.zeros((F, B))
    # /* Constants for calculation*/
    start_mel = 2595.0 * np.log10(1.0 + StFreq / 700)
    fs_per_2_mel = 2595.0 * np.log10(1.0 + (fs / 2) / 700)
    for b in range(B):
        # /* Calculating mel-scaled frequency and the corresponding FFT-bin */
        # /* number for the lower edge of the band                          */
        freq = 700 * (np.power(10.0, (start_mel + (b) / (B + 1) * (fs_per_2_mel - start_mel)) / 2595) - 1.0)
        f1 = (2 * F * freq / fs + 0.5)
        # /* Calculating mel-scaled frequency for the upper edge of the band */
        freq = 700 * (
        np.power(10.0, (start_mel + (b + 2) / (B + 1) * (fs_per_2_mel - start_mel)) / 2595) - 1.0)
        # /* Calculating and storing the length of the band in terms of FFT-bins*/
        f3 = (2 * F * freq / fs + 0.5)
        f2 = (f1 + f3) / 2
        f3 = min(f3, F - 1)
        s = 0.0
        f1 = int(f1)
        f2 = int(f2)
        f3 = int(f3)
        for f in range(f1, f2):
            fb[f, b] = f * (1 / (f2 - f1)) - f1 / (f2 - f1)
            s += fb[f, b]
        for f in range(f2, f3):
            fb[f, b] = f * (-1 / (f3 - f2)) + f3 / (f3 - f2)
            s += fb[f, b]
        for f in range(f1, f3):
            fb[f, b] /= s  # //normalization
    return fb

def f_base_dct(N):
    b = np.zeros((N, N))
    for n in range(N):
        if n == 0:
            kn = np.sqrt(1 / N)
        else:
            kn = np.sqrt(2 / N)
        for m in range(N):
            b[m, n] = kn * np.cos((2 * m + 1) * n * np.pi / (2 * N))
    return b


def frame_fb_mfcc(data, fs, B, w, nfft, max_windows):

    N =[int(wi * fs) for wi in w]
    F =[int(2 ** np.ceil(np.log2( Ni)))//2 for Ni in N]
    fb = [ fb_etsi(Fi, Bi, fs) for Fi, Bi in zip(F, B)]
    # print(f'fb es {len(fb[0])}')
    dct = [ f_base_dct(Bi) for Bi in B] 
    # print(f'dct es {len(dct[0])}')


    x = offset(data)
    x = preemphasis(x)
    
    X = hamming(windowing2(x, fs=fs, Ns=w[0], Ms=m, Mw=max_windows))
    
    Xfft = fft(X, nfft[0])
    Xb = np.log(Xfft.dot( fb[0] ) + 1)
    Xc = Xb.dot(dct[0])                                
    
    X = np.concatenate( [Xb, Xc], 1 )
    
    X = np.asarray(X, dtype=np.float32)
    
    # print("El tamaño de x08k2 es: ",XX.shape)
    # print("Vector con FilterBank MFCC del frame: \n", X[:10])
    
    return X

def read_pkl(f):
    with open(f, 'rb') as file:  # Open the file in binary read mode
        return pickle.load(file)

def norm_fb_frame(frame_fbmfcc):
    # Normalization of fbmfcc
    file = workspace_dir + 'data/model/fe1_norm1.pkl'  # de donde salen??
    x = frame_fbmfcc
    mu, std = read_pkl(file)
    x -= mu
    x /= std + 1e-6
    # print("Vector con FB MFCC normalizado del frame: \n", x[:10])

    return x
    

In [5]:
# Load model dimensions and weights

def load_obj(file):
    if not isinstance(file,str):
        return pickle.load(f)

    root,ext = os.path.splitext(file)
    if ext == '.gz':
        with gzip.open(file, 'rb') as f:
            return pickle.load(f)
    else:
        with open(file, 'rb') as f:
            return pickle.load(f)

input_dim, output_dim = load_obj(workspace_dir + 'data/model/dimensions.pkl') 

print('  input_dim: %s' % str(input_dim))
print('  output_dim: %s' % str(output_dim))


import sys
sys.path.append( workspace_dir + 'src/net')

from net_snr import Net_snr

net_snr = Net_snr(input_dim, output_dim, cuda=False)
net_snr.load_theta( workspace_dir + 'data/model/theta_last')

  input_dim: 576
  output_dim: 512

  Net_snr:
    nb_params: 29.99M
    cuda: False
    float16: False
    single_gpu: True
    opt: adama, 0.001, None
Adam (
Parameter Group 0
    amsgrad: True
    betas: (0.9, 0.999)
    eps: 1e-08
    lr: 0.001
    weight_decay: 0
)
    reading obj /home/adelval/BTS/TFM/test/data/model/theta_last


In [6]:
# Model inference
def net_eval(frame_concat):

    net_snr.set_mode_train(False)

    x = frame_concat
    # x = x.reshape(1, -1)
    # print(x.shape)

    snr = net_snr.predict(x)
    snr = to_numpy(snr.squeeze())
    print(f'La máscara del frame caculado es de {snr.shape}')
    
    #scipy.io.savemat(f, mdict={'snr': snr})
    # x = to_numpy(x.squeeze())
    # snr = to_numpy(snr.squeeze())
    return snr


In [7]:
# Aplicar mascara a frame 

def apply_filter(data, filt, frame=640, shift=160, nfft=1024):

    win = np.sqrt(np.hanning(frame))
    win = np.array(win, dtype=np.float32)
    
    yw = np.zeros(data.size)
    print(data.size)
    #yw[0:frame] = data[0:frame]*win*1e-3
    #it = int(np.floor((data.size-frame)/shift))
    #print(f'La ventana se desplazara {it} veces')
    it = 1
    for i in range(0,it):
        print(f'El frame sin enventanado con rn resulta {data[:10]}')
        xw = data[i*shift : i*shift+frame] * win
        print(f'El frame enventanado resulta {xw[:10]}')
        Xfft = np.fft.fft(xw, nfft)
        Xfft = Xfft[0:int(nfft/2+1)]
        print(Xfft.shape)
        print(f'El tamaño de la transformada del frame es {Xfft.shape}')
        print(f'La transformada del frame es {Xfft[:10]}')
        outf = Xfft * filt[:,i]
        print(f'El tamaño de la salida del filtro sera {outf.shape}')
        print(f'La salida del filtro es {outf[:10]}')
        fliped = np.flip(np.conj(outf[1:int(nfft/2)]), axis=0)
        outw = np.concatenate((outf, fliped), axis=0)
        outw = np.real(np.fft.ifft(outw, nfft, axis=0))
        print(f'El frame mejorado resulta sin OLA es {outw[:10]}')
        #print(f'La salida de la ifft tendra {outw.shape} samples') # de las que nos quedamos con 640 porque el resto son relleno
        yw[i*shift : i*shift+frame] = yw[i*shift : i*shift+frame] + outw[0:frame]*win
        print(f'El frame mejorado resulta {yw[:10]}')

    #yw[it*shift+frame:] = data[it*shift+frame:]*1e-3

    return yw

# ----------------------------------------------------------------------------------------------------------------------
def noiseReduction(data, snr_net, fs, frame, shift, nfft, gmin):
    
    # Add un-audible noise to avoid signals with 0
    data = data + 1e-7*np.random.rand(data.size) 
    snr_net = snr_net.reshape(-1,1) # para darle 2-D
    # Load snr_net
    snr_net = np.concatenate((snr_net, snr_net[int(nfft/2-1):, :]), axis=0)
    #snr_net = np.concatenate((snr_net, snr_net[int(nfft/2-1):, :]))
    # VAD
    ini = int(np.floor((nfft*300)/fs))
    out = int(np.ceil((nfft*2500)/fs))
    E = np.mean(snr_net[ini:out,:], axis=0)
    vad = 1/(1 + np.exp(-90*(E-0.05)))

    difference = 1 - snr_net
    difference[difference < 1e-7] = 1e-7
    gamma = 1 / difference
    upsilon = gamma * snr_net
    g = np.exp(0.5 * sc.exp1(upsilon)) * snr_net
    g[g > 1] = 1
    gmin = gmin/2
    gtotal = np.power(g, snr_net) * np.power((gmin * snr_net), (1 - snr_net))
    filt = np.power(gtotal,vad) * np.power(1e-3,1-vad)
    print(f'El filtro será {filt[:10]}')
    yw = apply_filter(data, filt, frame, shift, nfft)

    return yw, filt


In [18]:
# x_test = ['/home/adelval/BTS/TFM/audios/audio_1.wav']
x_test = ['/home/adelval/BTS/TFM/audios/7-CH0_C01_city_5dB.wav']
# x_test = ['/home/adelval/BTS/TFM/test/data/audio/minitest_16k/10-CH0_C01_airport_10dB.wav']
# x_test = ['/home/adelval/BTS/TFM/afterburner8k/data/audio/minitest_8k/5-CH0_C01_stadium_15dB.wav']
# x_test = ['/home/adelval/BTS/TFM/afterburner_2023/afterburner16k/data/audio/callcenter_movistar/AUDIOS/Audio-AudioModule_708351_AudioChannel_22403522_19-May-2023_15.53.49.097.wav']
print(f'El audio elegido es {x_test[0]}')

output_enh = os.path.basename(x_test[0])
output_enh = "/home/adelval/BTS/TFM/audios/enh/"+output_enh.replace('.wav','_f2f_enh_4to20w.wav')
print(f'El audio mejorado es {output_enh}')

El audio elegido es /home/adelval/BTS/TFM/audios/7-CH0_C01_city_5dB.wav
El audio mejorado es /home/adelval/BTS/TFM/audios/enh/7-CH0_C01_city_5dB_f2f_enh_4to20w.wav


In [19]:
audio, fs = read_audio(x_test[0])
print(f'La frecuencia de muestreo es {fs} Hz')
print(f'La duración del audio es {len(audio)/fs} segundos y {len(audio)} muestras')

# Parámetros
#fs=16000
B=[32]
w=[0.040]
m=0.010
nfft=[1024]
gmin = 0.0562


frame_size = 0.01  # 10 ms
frame_samples = int(frame_size * fs)  # Muestras por frame
print(f'Un frame tiene una duración de {frame_samples} samples')
shift_size = 0.01  # 10 ms
shift_samples = int(shift_size * fs)  # Desplazamiento entre frames 160
print(f'Desplazamiento de {shift_samples} samples')
window_size = 0.04  # 40 ms (640 muestras)
window_samples = int(window_size * fs)  # Muestras por ventana 640
print(f'La ventana tiene una duración de {window_samples} samples')
min_windows = 4
max_windows = 20
window_inference_min = int((min_windows + (window_size/frame_size)-1) * frame_samples)
window_inference_max = int((max_windows + (window_size/frame_size)-1) * frame_samples)
print(f'El buffer retendra hasta {window_inference_max} samples')
buffer_frame = np.zeros(0)  # Buffer de ventana recibida

it = 0
snr_frame_mask = np.ones((512,min_windows)) # Inicializado con la duración de la ventana de inferencia
yenh = np.zeros(len(audio)) # Inicializado con la duración del audio original

# CALCULO DE LA MÁSCARA SNR 
for n_frame in range(int((len(audio)/fs)*100)):
    # Obtener el frame de audio
    frame = audio[n_frame * shift_samples: n_frame * shift_samples + frame_samples]
    
    # Concatenar el frame recibido al buffer `buffer_frame`
    buffer_frame = np.concatenate([buffer_frame, frame])

    if len(buffer_frame) >= window_samples:
        # Extraer las primeras 640 muestras como ventana completa
        if len(buffer_frame) < window_inference_max:
            work_window = buffer_frame[it*shift_samples:it*shift_samples+window_samples]
            print("Iteración número:", n_frame)
            print("Ventana acumulada", work_window[:10])
            print("Ventana acumulada", work_window[160:170])
            print("Ventana acumulada", work_window[320:330])
            print("Ventana acumulada", work_window[480:490])
            it += 1 # Number of windows received
            if len(buffer_frame) >= window_inference_min:
                print("Reached MIN WINDOW --> STRATING INFERENCE")
                work_inf_frames = buffer_frame[:window_inference_max]
                fft_windows = frame_fft(work_inf_frames, fs, w, nfft, it)
                fft_windows_log = log_scale(fft_windows)
                fb_windows = frame_fb_mfcc(work_inf_frames, fs, B, w, nfft, it)
                fb_windows_norm = norm_fb_frame(fb_windows)
                windows_concat = np.concatenate( (fft_windows_log,fb_windows_norm), 1 )
                snr_frame_mask = net_eval(windows_concat)
                snr_frame_mask = snr_frame_mask.T
                print(f'La máscara {it} calculada es de {snr_frame_mask.shape}')
                print(f'Frames restantes en el buffer {len(buffer_frame)/frame_samples}')



        # Si el buffer alcanza o excede las 20 ventanas para hacer la inferencia
        else:
            work_window = buffer_frame[window_inference_max-window_samples:] # Los últimos frames del buffer
            print("Iteración número:", n_frame)
            # print("Ventana acumulada", work_window[:10])
            # print("Ventana acumulada", work_window[160:170])
            # print("Ventana acumulada", work_window[320:330])
            # print("Ventana acumulada", work_window[480:490])
            print("Reached MAX WINDOW --> Starting Inference")
            work_inf_frames = buffer_frame[:window_inference_max]
            # print(f'0 frame  {work_inf_frames[:10]}')
            # print(f'1 frame  {work_inf_frames[frame_samples:frame_samples+10]}')
            # print(f'2 frame  {work_inf_frames[2*frame_samples:2*frame_samples+10]}')
            # print(f'3 frame  {work_inf_frames[3*frame_samples:3*frame_samples+10]}')
            # print(f'23 frame {work_inf_frames[22*frame_samples:22*frame_samples+10]}')

            fft_windows = frame_fft(work_inf_frames, fs, w, nfft, max_windows)
            # print("0 Vector 2D con FFT para cada frame por row \n",  fft_windows[0,:10])
            # print("1 Vector 2D con FFT para cada frame por row \n",  fft_windows[1,:10])
            # print("2 Vector 2D con FFT para cada frame por row \n",  fft_windows[2,:10])
            # print("3 Vector 2D con FFT para cada frame por row \n",  fft_windows[3,:10])
            # print("20 Vector 2D con FFT para cada frame por row \n", fft_windows[19,:10])
            fft_windows_log = log_scale(fft_windows)
            # print(" 0 Vector 2D con FFT en escala log para cada frame por row \n",  fft_windows_log[0,:10])
            # print(" 1 Vector 2D con FFT en escala log para cada frame por row \n",  fft_windows_log[1,:10])
            # print(" 2 Vector 2D con FFT en escala log para cada frame por row \n",  fft_windows_log[2,:10])
            # print(" 3 Vector 2D con FFT en escala log para cada frame por row \n",  fft_windows_log[3,:10])
            # print("20 Vector 2D con FFT en escala log para cada frame por row \n", fft_windows_log[19,:10])
            fb_windows = frame_fb_mfcc(work_inf_frames, fs, B, w, nfft, max_windows)
            # print(fb_windows.shape)
            # print(f'0 Filtro Mel {fb_windows[0,:5]}')
            # print(f'1 Filtro Mel {fb_windows[1,:5]}')
            # print(f'2 Filtro Mel {fb_windows[2,:5]}')
            # print(f'3 Filtro Mel {fb_windows[3,:5]}')
            # print(f'19 Filtro Mel {fb_windows[19,:5]}')
            fb_windows_norm = norm_fb_frame(fb_windows)
            # print(f'0 Filtro Mel normalizado {fb_windows_norm[0,:5]}')
            # print(f'1 Filtro Mel normalizado {fb_windows_norm[1,:5]}')
            # print(f'2 Filtro Mel normalizado {fb_windows_norm[2,:5]}')
            # print(f'3 Filtro Mel normalizado {fb_windows_norm[3,:5]}')
            # print(f'19 Filtro Melnormalizado {fb_windows_norm[19,:5]}')
            windows_concat = np.concatenate( (fft_windows_log,fb_windows_norm), 1 )
            # print("El tamaño de x08k tras la concatenacion es: ",windows_concat.shape)
            # print(f'0 La concatenacion resulta  {windows_concat[0,:5]}  y {windows_concat[0,512:517]}')
            # print(f'1 La concatenacion resulta  {windows_concat[1,:5]}  y {windows_concat[1,512:517]}')
            # print(f'2 La concatenacion resulta  {windows_concat[2,:5]}  y {windows_concat[2,512:517]}')
            # print(f'3 La concatenacion resulta  {windows_concat[3,:5]}  y {windows_concat[3,512:517]}')
            # print(f'19 La concatenacion resulta {windows_concat[19,:5]} y {windows_concat[19,512:517]}')

            snr_frame_mask = net_eval(windows_concat)
            # print(snr_frame_mask.shape)
            # print(snr_frame_mask[0,:10])
            # print(snr_frame_mask[1,:10])
            # print(snr_frame_mask[2,:10])
            # print(snr_frame_mask[3,:10])
            # print(snr_frame_mask[19,:10])
            snr_frame_mask = snr_frame_mask.T
            #Desplazar las muestras en `buffer_frame` para la próxima ventana
            buffer_frame = buffer_frame[shift_samples:]
            print(f'Frames restantes en el buffer {len(buffer_frame)/frame_samples}')

        # Aqui haría la evaluacion con la máscara pertinente (para las primeras 3 ventanas sin máscara calculada)
        cnt = n_frame - 3
        print(f'EVALUATION OF WINDOW {cnt}')
        x = np.array(work_window, dtype=np.float32) / 2 ** 15 # 0.04 * fs = 640 samples
        print(f'El frame sin enventanado resulta {x[:10]}')
        print(f'Se le aplica la mascara {snr_frame_mask[:10,-1]}')
        xenh, filt = noiseReduction(x, snr_frame_mask[:,-1], fs, window_samples, shift_samples, nfft[0], gmin)
        print(f'Las dimensiones del filtro son {filt.shape}')
        print(f'Window processed {cnt} {xenh.shape} {yenh[cnt*shift_samples : cnt*shift_samples+window_samples].shape}')
        yenh[cnt*shift_samples : cnt*shift_samples+window_samples] = yenh[cnt*shift_samples : cnt*shift_samples+window_samples] + xenh[0:window_samples]
        
print(f'el audio tiene len {len(yenh)}')
wavfile.write(output_enh,fs,yenh)



# print(f'0 frame  {work_inf_frames[:10]}')
# print(f'1 frame  {work_inf_frames[frame_samples:frame_samples+10]}')
# print(f'2 frame  {work_inf_frames[2*frame_samples:2*frame_samples+10]}')
# print(f'3 frame  {work_inf_frames[3*frame_samples:3*frame_samples+10]}')
# print(f'23 frame {work_inf_frames[22*frame_samples:22*frame_samples+10]}')


La frecuencia de muestreo es 16000 Hz
La duración del audio es 4.1100625 segundos y 65761 muestras
Un frame tiene una duración de 160 samples
Desplazamiento de 160 samples
La ventana tiene una duración de 640 samples
El buffer retendra hasta 3680 samples
Iteración número: 3
Ventana acumulada [3548. 3653. 2920. 3812. 2842. 2843. 1005.  925. 1277. 1376.]
Ventana acumulada [ -706. -1383. -1530. -3424. -3447. -3115. -3242. -1602. -2239. -2796.]
Ventana acumulada [ 1608.   508.   115.   457.    59.  1001.   290. -1480.  -670. -1660.]
Ventana acumulada [   53.  -120.  -717. -1116. -2995. -2728. -4247. -4399. -4017. -4644.]
EVALUATION OF WINDOW 0
El frame sin enventanado resulta [0.10827637 0.11148071 0.08911133 0.11633301 0.08673096 0.08676147
 0.03067017 0.02822876 0.03897095 0.04199219]
Se le aplica la mascara [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
El filtro será [[1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]]
640
El frame sin enventanado con rn resulta [0.10827638 0.11148072 0.08911